# Codelion PTS steering-vector analysis

Single notebook covering:

1. Download `codelion/Qwen3-0.6B-pts-steering-vectors` and cache under `data/codelion_pts/`.
2. EDA on the dataset schema and distributions.
3. Visualization: heatmap, PCA, t-SNE, HDBSCAN re-clustering.
4. Qwen3-0.6B model + hook + parser scaffolding.
5. 8 steering tasks, each with `mean` + `top1` + `top2` arms (norm-matched). Each task is followed by a token-analysis block.
6. Comparison table + export zip.

A top-of-notebook `QUICK` flag reduces runtime to ~45 min for a first pass.


## 1. Install

In [ ]:
# Run once per runtime. Safe to skip if already installed.
%pip install -q "transformers>=4.44" datasets accelerate matplotlib seaborn scikit-learn tqdm pandas
%pip install -q hdbscan pyarrow


## 2. Imports + seed

In [ ]:
import json
import os
import random
import re
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
    DTYPE = torch.float32
else:
    DEVICE = "cpu"
    DTYPE = torch.float32

RUN_TAG = "codelion"
RESULTS_DIR = Path("nb_results") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"device={DEVICE} dtype={DTYPE} run_tag={RUN_TAG}")
print(f"results -> {RESULTS_DIR.resolve()}")


## 3. Config + QUICK toggle

In [ ]:
QUICK = True  # False for full run
MAX_EXAMPLES = 50 if QUICK else 100
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.6
TOP_P = 0.9
FACTORS = [1.2] if QUICK else [1.2, 2.0]
TOP_K_ARMS = (1,) if QUICK else (1, 2)
CANDIDATE_LAYERS = [14] if QUICK else [8, 14, 16]
DATA_DIR = Path("data/codelion_pts")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FORCE_RERUN = False
print(f"QUICK={QUICK} MAX_EXAMPLES={MAX_EXAMPLES} FACTORS={FACTORS} TOP_K_ARMS={TOP_K_ARMS} LAYERS={CANDIDATE_LAYERS}")


## 4. Download codelion dataset

In [ ]:
import pandas as pd
from datasets import load_dataset

parquet_path = DATA_DIR / "codelion_pts.parquet"
if parquet_path.exists() and not FORCE_RERUN:
    df = pd.read_parquet(parquet_path)
    print(f"Loaded cached parquet ({len(df)} rows)")
else:
    ds = load_dataset("codelion/Qwen3-0.6B-pts-steering-vectors", split="train")
    df = ds.to_pandas()
    df.to_parquet(parquet_path, index=False)
    df.head(3).to_json(DATA_DIR / "schema_preview.json", orient="records", indent=2)
    print(f"Downloaded {len(df)} rows; saved to {parquet_path}")

print("columns:", list(df.columns))
print("dtypes:\n", df.dtypes)


## 4b. Coerce vector columns to np.ndarrays (they may arrive as lists)

In [ ]:
def _as_array(x):
    if isinstance(x, np.ndarray):
        return x.astype(np.float32, copy=False)
    return np.asarray(x, dtype=np.float32)

df["steering_vector"] = df["steering_vector"].map(_as_array)
if "cluster_vector" in df.columns:
    df["cluster_vector"] = df["cluster_vector"].map(_as_array)
df["abs_prob_delta"] = df["prob_delta"].abs()

hidden_dim = int(df["steering_vector"].iloc[0].shape[0])
print(f"rows={len(df)} hidden_dim={hidden_dim}")
print(df[["reasoning_pattern", "prob_delta", "pivot_token"]].head())


## 5a. EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["figure.dpi"] = 110
VIZ_DIR = RESULTS_DIR / "viz"; VIZ_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
rp_counts = df["reasoning_pattern"].value_counts()
axes[0, 0].barh(rp_counts.index[::-1], rp_counts.values[::-1])
axes[0, 0].set_title("Rows per reasoning_pattern")

axes[0, 1].hist(df["prob_delta"], bins=40, color="#4c72b0")
axes[0, 1].axvline(0, color="k", lw=0.5)
axes[0, 1].set_title("prob_delta distribution (signed)")

axes[1, 0].hist(df["abs_prob_delta"], bins=40, color="#c44e52")
axes[1, 0].set_title("|prob_delta| distribution")

top_tok = df.groupby("pivot_token")["abs_prob_delta"].median().sort_values(ascending=False).head(20)
axes[1, 1].barh(top_tok.index[::-1], top_tok.values[::-1])
axes[1, 1].set_title("Top-20 pivot tokens by median |prob_delta|")
fig.tight_layout()
fig.savefig(VIZ_DIR / "eda.png"); plt.show()

if "cluster_id" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    ctab = pd.crosstab(df["cluster_id"], df["reasoning_pattern"])
    ctab.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title("cluster_id x reasoning_pattern")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(VIZ_DIR / "cluster_vs_pattern.png"); plt.show()


### Aside: what is `cluster_id` / `cluster_vector`?

Each row's `steering_vector` is a 1024-d direction derived from one pivot token in one generation. Raw rows are noisy. The PTS authors group similar rows (k-means-style) into clusters; `cluster_id` is the group assignment and `cluster_vector` is the cluster's centroid vector. Using `cluster_vector` is intended to be a denoised version of the direction shared by rows inside a cluster.

## 5a-viz. Steering-vector visualization

In [ ]:
import hdbscan
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score

# Stack 1024-d vectors into a matrix.
V = np.stack(df["steering_vector"].values)  # (N, 1024)
print(f"V shape={V.shape}")

# ---- 1. Heatmap (row-sorted, col 1-D PCA ordered, percentile-clipped).
sort_cols = [c for c in ["cluster_id", "reasoning_pattern", "prob_delta"] if c in df.columns]
order = df.sort_values(sort_cols).index.to_numpy()
V_sorted = V[order]

col_pca = PCA(n_components=1, random_state=SEED)
col_scores = col_pca.fit_transform(V_sorted.T).ravel()
col_order = np.argsort(col_scores)
V_heat = V_sorted[:, col_order]
lo, hi = np.percentile(V_heat, [1, 99])

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(V_heat, aspect="auto", cmap="RdBu_r", vmin=lo, vmax=hi)
plt.colorbar(im, ax=ax)
if "cluster_id" in df.columns:
    # cluster boundaries in the sorted order
    sorted_cids = df.loc[order, "cluster_id"].values
    boundaries = np.where(np.diff(sorted_cids) != 0)[0]
    for b in boundaries:
        ax.axhline(b + 0.5, color="black", lw=0.3, alpha=0.6)
ax.set_title("Steering vectors (rows grouped by cluster_id, cols 1-D PCA ordered)")
ax.set_xlabel("hidden dim (reordered)")
ax.set_ylabel("row (sorted)")
fig.tight_layout()
fig.savefig(VIZ_DIR / "viz_heatmap.png"); plt.show()


In [ ]:
# ---- 2. PCA 2-D.
pca2 = PCA(n_components=2, random_state=SEED)
P = pca2.fit_transform(V)
print("PCA explained variance:", pca2.explained_variance_ratio_.tolist())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette_rp = sns.color_palette("tab20", n_colors=df["reasoning_pattern"].nunique())
for color, (rp, idxs) in zip(palette_rp, df.groupby("reasoning_pattern").groups.items()):
    idxs = np.array(list(idxs))
    axes[0].scatter(P[idxs, 0], P[idxs, 1], s=10, alpha=0.6, label=str(rp)[:20], color=color)
axes[0].set_title("PCA colored by reasoning_pattern")
axes[0].legend(loc="best", fontsize=7, markerscale=1.5, ncol=2)

if "cluster_id" in df.columns:
    palette_cl = sns.color_palette("tab20", n_colors=df["cluster_id"].nunique())
    for color, (cid, idxs) in zip(palette_cl, df.groupby("cluster_id").groups.items()):
        idxs = np.array(list(idxs))
        axes[1].scatter(P[idxs, 0], P[idxs, 1], s=10, alpha=0.6, color=color)
    # Overlay cluster centroids.
    if "cluster_vector" in df.columns:
        C = np.stack(df.drop_duplicates("cluster_id")["cluster_vector"].values)
        CP = pca2.transform(C)
        axes[1].scatter(CP[:, 0], CP[:, 1], marker="*", s=180, edgecolor="black", facecolor="yellow", label="cluster_vector")
        axes[1].legend()
    axes[1].set_title("PCA colored by cluster_id (stars = cluster_vector)")
fig.tight_layout()
fig.savefig(VIZ_DIR / "viz_pca.png"); plt.show()


In [ ]:
# ---- 3. t-SNE 2-D.
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED, max_iter=750)
T = tsne.fit_transform(V)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
palette_rp = sns.color_palette("tab20", n_colors=df["reasoning_pattern"].nunique())
for color, (rp, idxs) in zip(palette_rp, df.groupby("reasoning_pattern").groups.items()):
    idxs = np.array(list(idxs))
    axes[0].scatter(T[idxs, 0], T[idxs, 1], s=10, alpha=0.6, label=str(rp)[:20], color=color)
axes[0].set_title("t-SNE colored by reasoning_pattern")
axes[0].legend(loc="best", fontsize=7, ncol=2)

if "cluster_id" in df.columns:
    palette_cl = sns.color_palette("tab20", n_colors=df["cluster_id"].nunique())
    for color, (cid, idxs) in zip(palette_cl, df.groupby("cluster_id").groups.items()):
        idxs = np.array(list(idxs))
        axes[1].scatter(T[idxs, 0], T[idxs, 1], s=10, alpha=0.6, color=color)
    axes[1].set_title("t-SNE colored by cluster_id")
fig.tight_layout()
fig.savefig(VIZ_DIR / "viz_tsne.png"); plt.show()


In [ ]:
# ---- 4. HDBSCAN re-clustering.
clusterer = hdbscan.HDBSCAN(min_cluster_size=20, metric="euclidean")
hdb_labels = clusterer.fit_predict(V)
noise_rate = float((hdb_labels == -1).mean())
print(f"HDBSCAN: {len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)} clusters, noise_rate={noise_rate:.3f}")

ari = None
if "cluster_id" in df.columns:
    ari = float(adjusted_rand_score(df["cluster_id"].values, hdb_labels))
    print(f"ARI(HDBSCAN, provided cluster_id) = {ari:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
mask = hdb_labels != -1
axes[0].scatter(P[~mask, 0], P[~mask, 1], s=8, c="lightgray", alpha=0.5, label="noise")
sc = axes[0].scatter(P[mask, 0], P[mask, 1], s=10, c=hdb_labels[mask], cmap="tab20", alpha=0.7)
axes[0].set_title(f"HDBSCAN labels on PCA plane (noise={noise_rate:.0%})")
axes[0].legend()

if "cluster_id" in df.columns:
    provided = df["cluster_id"].astype("category")
    ct = pd.crosstab(provided, pd.Series(hdb_labels, name="hdbscan"))
    sns.heatmap(ct, cmap="Blues", ax=axes[1], cbar=True, annot=False)
    axes[1].set_title(f"Contingency: provided vs HDBSCAN (ARI={ari:.3f})")
fig.tight_layout()
fig.savefig(VIZ_DIR / "viz_hdbscan.png"); plt.show()

viz_summary = {
    "pca_explained_variance": pca2.explained_variance_ratio_.tolist(),
    "tsne_kl_divergence": float(getattr(tsne, "kl_divergence_", 0.0)),
    "hdbscan_ari_vs_provided": ari,
    "hdbscan_noise_rate": noise_rate,
    "hdbscan_n_clusters": int(len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)),
}
(VIZ_DIR / "viz_summary.json").write_text(json.dumps(viz_summary, indent=2))
print(viz_summary)


## 5b. Qwen + hooks + parser + dataset

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"

_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(
    f"Loaded {MODEL_NAME} on {DEVICE} ({DTYPE}) "
    f"in {time.time() - _t0:.1f}s | hidden_size={model.config.hidden_size} "
    f"layers={model.config.num_hidden_layers}"
)


In [ ]:
assert hidden_dim == model.config.hidden_size, f"codelion hidden_dim {hidden_dim} != model hidden_size {model.config.hidden_size}"


In [ ]:
import torch.nn as nn


def _get_decoder_layer(mdl: nn.Module, layer_idx: int) -> nn.Module:
    if hasattr(mdl, "model") and hasattr(mdl.model, "layers"):
        return mdl.model.layers[layer_idx]
    if hasattr(mdl, "transformer") and hasattr(mdl.transformer, "h"):
        return mdl.transformer.h[layer_idx]
    if hasattr(mdl, "decoder") and hasattr(mdl.decoder, "layers"):
        return mdl.decoder.layers[layer_idx]
    raise AttributeError(f"Cannot find decoder layers in {type(mdl)}")


def _make_steering_hook(vector: torch.Tensor, strength: float):
    def hook(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        v = vector.to(hidden.device).to(hidden.dtype)
        if v.dim() == 1:
            v = v.view(1, 1, -1)
        delta = strength * v
        if isinstance(output, tuple):
            return (hidden + delta,) + output[1:]
        return hidden + delta
    return hook


def register_steering(mdl, layer: int, vector: torch.Tensor, strength: float):
    return [
        _get_decoder_layer(mdl, layer).register_forward_hook(
            _make_steering_hook(vector, strength)
        )
    ]


def register_additive_hooks(mdl, injections):
    '''Register one additive hook per (layer_idx, vector, strength) tuple.'''
    handles = []
    for layer_idx, vector, strength in injections:
        handles.append(
            _get_decoder_layer(mdl, layer_idx).register_forward_hook(
                _make_steering_hook(vector, strength)
            )
        )
    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()


In [ ]:
def extract_gsm8k_answer(text: str):
    m = re.search(r"####\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else None


def is_correct(pred, gt):
    if pred is None or gt is None:
        return False
    p = pred.strip().replace(",", "")
    g = gt.strip().replace(",", "")
    try:
        return float(p) == float(g)
    except ValueError:
        return p == g


def compute_metrics(results):
    n = len(results)
    if n == 0:
        return {"accuracy": 0.0, "f1": 0.0, "parse_rate": 0.0, "correct": 0, "n": 0}
    correct = sum(1 for r in results if r["correct"])
    parsed = sum(1 for r in results if r["predicted"] is not None)
    tp = correct
    fp = parsed - correct
    fn = n - correct
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        "accuracy": correct / n,
        "f1": f1,
        "parse_rate": parsed / n,
        "correct": correct,
        "n": n,
    }


# Inline parser sanity checks (mirror tests/test_answer_extraction.py).
assert extract_gsm8k_answer("#### 42") == "42"
assert extract_gsm8k_answer("He has #### 1,234 apples.") == "1234"
assert extract_gsm8k_answer("The net change is #### -7") == "-7"
assert extract_gsm8k_answer("So the answer is #### 3.50") == "3.50"
assert extract_gsm8k_answer("...so the total is 18 apples.") == "18"
assert extract_gsm8k_answer("") is None
assert is_correct("3.0", "3") is True
assert is_correct("1234", "1,234") is True
assert is_correct(None, "5") is False
print("Parser sanity checks passed.")


In [ ]:
from datasets import load_dataset

_ds_full = load_dataset("openai/gsm8k", "main", split="test")
_rng = np.random.default_rng(SEED)
_indices = sorted(
    _rng.choice(len(_ds_full), size=min(MAX_EXAMPLES, len(_ds_full)), replace=False).tolist()
)
ds_subset = _ds_full.select(_indices)
examples = [
    {"question": row["question"],
     "ground_truth": extract_gsm8k_answer(row["answer"]),
     "answer_full": row["answer"]}
    for row in ds_subset
]
print(f"GSM8K test subset: {len(examples)} examples (seed={SEED})")


In [ ]:
def generate_once(prompt: str) -> str:
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def _reseed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def run_eval_with_hooks(label, register_fn, desc_prefix=""):
    '''register_fn(model) -> list of hook handles (or []). Runs paired GSM8K subset.'''
    _reseed()
    handles = register_fn(model) if register_fn is not None else []
    results = []
    try:
        for i, ex in enumerate(tqdm(examples, desc=f"{desc_prefix}{label}")):
            prompt = f"Question: {ex['question']}\n\nLet's think step by step.\n\n"
            text = generate_once(prompt)
            pred = extract_gsm8k_answer(text)
            ok = is_correct(pred, ex["ground_truth"])
            results.append({
                "idx": i,
                "question": ex["question"],
                "ground_truth": ex["ground_truth"],
                "predicted": pred,
                "correct": ok,
                "full_output": text[-1000:],
            })
    finally:
        remove_hooks(handles)
    return results, compute_metrics(results)


def save_run(label, results, metrics, extra_state=None):
    (RESULTS_DIR / f"{label}_results.json").write_text(json.dumps(results, indent=2))
    (RESULTS_DIR / f"{label}_generations.txt").write_text(
        "\n\n====\n\n".join(
            f"[{r['idx']}] gt={r['ground_truth']} pred={r['predicted']} correct={r['correct']}\n{r['full_output']}"
            for r in results
        )
    )
    state = {
        "label": label,
        "model": MODEL_NAME,
        "seed": SEED,
        "max_examples": len(examples),
        "max_new_tokens": MAX_NEW_TOKENS,
        "device": DEVICE,
        "run_tag": RUN_TAG,
        "metrics": metrics,
    }
    if extra_state:
        state.update(extra_state)
    (RESULTS_DIR / f"{label}_state.json").write_text(json.dumps(state, indent=2))
    return state


### Reusable eval + token-analysis helpers

In [ ]:
TASKS_ROOT = RESULTS_DIR / "tasks"; TASKS_ROOT.mkdir(exist_ok=True)
all_task_metrics = []  # flat list of dicts, one per (task, group, arm, factor)

def _vec_from_rows(rows_df, agg, k=None, sort_key="abs_prob_delta"):
    '''Return (aggregated vector np.ndarray, used_row_ids list).'''
    if agg == "mean":
        v = np.stack(rows_df["steering_vector"].values).mean(axis=0)
        ids = rows_df.index.tolist()
    elif agg == "weighted_mean_signed":
        w = rows_df["prob_delta"].values.astype(np.float32)
        V = np.stack(rows_df["steering_vector"].values)
        v = (V * w[:, None]).sum(axis=0) / max(1e-8, np.abs(w).sum())
        ids = rows_df.index.tolist()
    elif agg == "weighted_mean_abs":
        w = rows_df["abs_prob_delta"].values.astype(np.float32)
        V = np.stack(rows_df["steering_vector"].values)
        v = (V * w[:, None]).sum(axis=0) / max(1e-8, w.sum())
        ids = rows_df.index.tolist()
    elif agg in ("top1", "top2"):
        kk = 1 if agg == "top1" else 2
        pick = rows_df.sort_values(sort_key, ascending=False).head(kk)
        V = np.stack(pick["steering_vector"].values)
        v = V.sum(axis=0)
        ids = pick.index.tolist()
    elif agg == "raw_centroid":
        v = rows_df["cluster_vector"].iloc[0]
        ids = [int(rows_df.index[0])]
    else:
        raise ValueError(f"unknown agg={agg}")
    return v.astype(np.float32), ids


def evaluate_with_vectors(rows_df, layer, factor, label, agg="mean",
                           sort_key="abs_prob_delta", match_norm_to=None, extra=None):
    cache_dir = TASKS_ROOT / label.split("__")[0]
    cache_dir.mkdir(parents=True, exist_ok=True)
    res_path = cache_dir / f"{label}_results.json"
    strength = factor - 1.0
    vec, ids = _vec_from_rows(rows_df, agg=agg, sort_key=sort_key)
    if match_norm_to is not None:
        ref_norm = float(np.linalg.norm(match_norm_to))
        cur = float(np.linalg.norm(vec))
        if cur > 1e-8:
            vec = vec * (ref_norm / cur)
    vec_tensor = torch.from_numpy(vec).to(torch.float32)
    injected_norm = float(vec_tensor.norm().item())

    if res_path.exists() and not FORCE_RERUN:
        res = json.loads(res_path.read_text())
        met = compute_metrics(res)
    else:
        def reg(mdl, vt=vec_tensor, s=strength, lyr=layer):
            return register_steering(mdl, lyr, vt, s)
        res, met = run_eval_with_hooks(label, register_fn=reg)
        (cache_dir / f"{label}_results.json").write_text(json.dumps(res, indent=2))
        (cache_dir / f"{label}_generations.txt").write_text(
            "\n\n====\n\n".join(
                f"[{r['idx']}] gt={r['ground_truth']} pred={r['predicted']} correct={r['correct']}\n{r['full_output']}"
                for r in res
            )
        )
        state = {"label": label, "layer": layer, "factor": factor, "strength": strength,
                 "agg": agg, "injected_norm": injected_norm, "n_rows": int(len(rows_df)),
                 "used_row_ids": ids, "metrics": met}
        if extra:
            state.update(extra)
        (cache_dir / f"{label}_state.json").write_text(json.dumps(state, indent=2))
    return {"metrics": met, "results": res, "vector": vec, "injected_norm": injected_norm,
            "used_row_ids": ids, "agg": agg, "layer": layer, "factor": factor, "label": label}


import string

_STOPWORDS_RAW = "the a an of in to for and or is are was were be been being this that these those on at by with as it its i me my we our you your he she they them their what which who whom how why when where than then so if but not no yes do does did done have has had ll ve re d s t m"
STOPWORDS = set(_STOPWORDS_RAW.split())


def analyze_tokens(base_outputs, steered_outputs, task_id, top_n=20):
    def tokens_of(text_blob):
        text_blob = text_blob.lower()
        for ch in string.punctuation:
            text_blob = text_blob.replace(ch, " ")
        toks = text_blob.split()
        return [t for t in toks if t and not t.isdigit() and t not in STOPWORDS]

    from collections import Counter
    base_text = " ".join(r["full_output"] for r in base_outputs)
    steer_text = " ".join(r["full_output"] for r in steered_outputs)
    b = Counter(tokens_of(base_text))
    s = Counter(tokens_of(steer_text))
    total_b = sum(b.values()) or 1
    total_s = sum(s.values()) or 1
    words = set(b) | set(s)
    scored = []
    for w in words:
        fb = b[w] / total_b * 1e6
        fs = s[w] / total_s * 1e6
        if fb + fs < 5:
            continue
        scored.append((w, float(np.log((fs + 1) / (fb + 1))), fb, fs))
    promoted = sorted(scored, key=lambda x: -x[1])[:top_n]
    suppressed = sorted(scored, key=lambda x: x[1])[:top_n]

    out_dir = TASKS_ROOT / task_id; out_dir.mkdir(parents=True, exist_ok=True)
    payload = {"task_id": task_id, "promoted": promoted, "suppressed": suppressed}
    (out_dir / "token_analysis.json").write_text(json.dumps(payload, indent=2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].barh([w for w, *_ in promoted[::-1]], [lr for _, lr, *_ in promoted[::-1]], color="seagreen")
    axes[0].set_title(f"{task_id}: top promoted")
    axes[1].barh([w for w, *_ in suppressed[::-1]], [lr for _, lr, *_ in suppressed[::-1]], color="indianred")
    axes[1].set_title(f"{task_id}: top suppressed")
    fig.tight_layout()
    fig.savefig(out_dir / "token_analysis.png"); plt.close(fig)
    return payload


### Base eval (shared by all tasks)

In [ ]:
base_cache = RESULTS_DIR / "base_results.json"
if base_cache.exists() and not FORCE_RERUN:
    base_results = json.loads(base_cache.read_text())
    base_metrics = compute_metrics(base_results)
else:
    base_results, base_metrics = run_eval_with_hooks("base", register_fn=None)
    save_run("base", base_results, base_metrics)
print(f"base: acc={base_metrics['accuracy']:.3f}")


## 5c. Steering tasks

### Task 1. task_all_mean — pick BEST_LAYER

In [ ]:
TASK_ID = "task_all_mean"
layer_scores = {}
for L in CANDIDATE_LAYERS:
    key = f"{TASK_ID}__L{L}_f1.2"
    out = evaluate_with_vectors(df, layer=L, factor=1.2, label=key, agg="mean")
    all_task_metrics.append({"task": TASK_ID, "group": "all", "arm": "mean", "k": None,
                              "layer": L, "factor": 1.2,
                              **{k: out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": out["injected_norm"]})
    layer_scores[L] = out["metrics"]["accuracy"]
    analyze_tokens(base_results, out["results"], f"{TASK_ID}_L{L}")
BEST_LAYER = max(layer_scores, key=layer_scores.get)
print("layer_scores:", layer_scores, "-> BEST_LAYER =", BEST_LAYER)

if 2.0 in FACTORS:
    key = f"{TASK_ID}__L{BEST_LAYER}_f2.0"
    out = evaluate_with_vectors(df, layer=BEST_LAYER, factor=2.0, label=key, agg="mean")
    all_task_metrics.append({"task": TASK_ID, "group": "all", "arm": "mean", "k": None,
                              "layer": BEST_LAYER, "factor": 2.0,
                              **{k: out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": out["injected_norm"]})


### Task 2. task_delta_sign (pos/neg groups; mean + top1/top2 arms)

In [ ]:
TASK_ID = "task_delta_sign"
groups = {
    "pos": df[df["prob_delta"] > 0],
    "neg": df[df["prob_delta"] < 0],
}
for gname, gdf in groups.items():
    if len(gdf) == 0:
        continue
    sort_key = "prob_delta" if gname == "pos" else "abs_prob_delta"
    # Mean arm
    mean_out = evaluate_with_vectors(gdf, layer=BEST_LAYER, factor=1.2,
                                      label=f"{TASK_ID}__{gname}_mean_f1.2", agg="mean")
    mean_vec = mean_out["vector"]
    all_task_metrics.append({"task": TASK_ID, "group": gname, "arm": "mean", "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: mean_out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": mean_out["injected_norm"]})
    analyze_tokens(base_results, mean_out["results"], f"{TASK_ID}_{gname}_mean")
    for k in TOP_K_ARMS:
        arm = f"top{k}"
        out = evaluate_with_vectors(gdf, layer=BEST_LAYER, factor=1.2,
                                     label=f"{TASK_ID}__{gname}_{arm}_f1.2", agg=arm,
                                     sort_key=sort_key, match_norm_to=mean_vec)
        all_task_metrics.append({"task": TASK_ID, "group": gname, "arm": arm, "k": k,
                                  "layer": BEST_LAYER, "factor": 1.2,
                                  **{kk: out["metrics"][kk] for kk in ("accuracy", "f1", "parse_rate")},
                                  "injected_norm": out["injected_norm"]})
        analyze_tokens(base_results, out["results"], f"{TASK_ID}_{gname}_{arm}")


### Task 3. task_delta_magnitude (top/bottom 25% |prob_delta|)

In [ ]:
TASK_ID = "task_delta_magnitude"
qhigh = df["abs_prob_delta"].quantile(0.75)
qlow = df["abs_prob_delta"].quantile(0.25)
groups = {
    "high": df[df["abs_prob_delta"] >= qhigh],
    "low":  df[df["abs_prob_delta"] <= qlow],
}
for gname, gdf in groups.items():
    if len(gdf) == 0:
        continue
    mean_out = evaluate_with_vectors(gdf, layer=BEST_LAYER, factor=1.2,
                                      label=f"{TASK_ID}__{gname}_mean_f1.2", agg="mean")
    mean_vec = mean_out["vector"]
    all_task_metrics.append({"task": TASK_ID, "group": gname, "arm": "mean", "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: mean_out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": mean_out["injected_norm"]})
    analyze_tokens(base_results, mean_out["results"], f"{TASK_ID}_{gname}_mean")
    for k in TOP_K_ARMS:
        arm = f"top{k}"
        out = evaluate_with_vectors(gdf, layer=BEST_LAYER, factor=1.2,
                                     label=f"{TASK_ID}__{gname}_{arm}_f1.2", agg=arm,
                                     sort_key="abs_prob_delta", match_norm_to=mean_vec)
        all_task_metrics.append({"task": TASK_ID, "group": gname, "arm": arm, "k": k,
                                  "layer": BEST_LAYER, "factor": 1.2,
                                  **{kk: out["metrics"][kk] for kk in ("accuracy", "f1", "parse_rate")},
                                  "injected_norm": out["injected_norm"]})
        analyze_tokens(base_results, out["results"], f"{TASK_ID}_{gname}_{arm}")


### Task 4. task_reasoning_pattern (>=40 rows per group)

In [ ]:
TASK_ID = "task_reasoning_pattern"
eligible = df.groupby("reasoning_pattern").filter(lambda g: len(g) >= 40)
for rp, gdf in eligible.groupby("reasoning_pattern"):
    rp_slug = str(rp).replace(" ", "_")[:24]
    mean_out = evaluate_with_vectors(gdf, layer=BEST_LAYER, factor=1.2,
                                      label=f"{TASK_ID}__{rp_slug}_mean_f1.2", agg="mean")
    mean_vec = mean_out["vector"]
    all_task_metrics.append({"task": TASK_ID, "group": rp_slug, "arm": "mean", "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: mean_out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": mean_out["injected_norm"]})
    analyze_tokens(base_results, mean_out["results"], f"{TASK_ID}_{rp_slug}_mean")
    for k in TOP_K_ARMS:
        arm = f"top{k}"
        out = evaluate_with_vectors(gdf, layer=BEST_LAYER, factor=1.2,
                                     label=f"{TASK_ID}__{rp_slug}_{arm}_f1.2", agg=arm,
                                     sort_key="abs_prob_delta", match_norm_to=mean_vec)
        all_task_metrics.append({"task": TASK_ID, "group": rp_slug, "arm": arm, "k": k,
                                  "layer": BEST_LAYER, "factor": 1.2,
                                  **{kk: out["metrics"][kk] for kk in ("accuracy", "f1", "parse_rate")},
                                  "injected_norm": out["injected_norm"]})
        analyze_tokens(base_results, out["results"], f"{TASK_ID}_{rp_slug}_{arm}")


### Task 5. task_cluster_vectors (top-5 clusters by row count)

In [ ]:
TASK_ID = "task_cluster_vectors"
if "cluster_vector" in df.columns and "cluster_id" in df.columns:
    top_clusters = df["cluster_id"].value_counts().head(5).index.tolist()
    cluster_results = {}
    for cid in top_clusters:
        cdf = df[df["cluster_id"] == cid]
        out = evaluate_with_vectors(cdf.head(1), layer=BEST_LAYER, factor=1.2,
                                     label=f"{TASK_ID}__c{cid}_raw_f1.2", agg="raw_centroid")
        all_task_metrics.append({"task": TASK_ID, "group": f"c{cid}", "arm": "cluster_vector", "k": None,
                                  "layer": BEST_LAYER, "factor": 1.2,
                                  **{k: out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                                  "injected_norm": out["injected_norm"]})
        analyze_tokens(base_results, out["results"], f"{TASK_ID}_c{cid}")
        cluster_results[cid] = out["metrics"]["accuracy"]
    BEST_CLUSTER = max(cluster_results, key=cluster_results.get)
    print("cluster_results:", cluster_results, "-> BEST_CLUSTER =", BEST_CLUSTER)
else:
    print("No cluster columns in dataset; skipping task 5.")
    BEST_CLUSTER = None


### Task 6. task_cluster_vs_raw (cluster_vector vs mean vs top2 within best cluster)

In [ ]:
TASK_ID = "task_cluster_vs_raw"
if BEST_CLUSTER is not None:
    cdf = df[df["cluster_id"] == BEST_CLUSTER]
    # arm 1: cluster_vector (already done in task 5 — replay for fair comparison table)
    out_cv = evaluate_with_vectors(cdf.head(1), layer=BEST_LAYER, factor=1.2,
                                    label=f"{TASK_ID}__c{BEST_CLUSTER}_cluster_vector_f1.2", agg="raw_centroid")
    all_task_metrics.append({"task": TASK_ID, "group": f"c{BEST_CLUSTER}", "arm": "cluster_vector", "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: out_cv["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": out_cv["injected_norm"]})
    # arm 2: mean of raw rows in cluster
    out_mean = evaluate_with_vectors(cdf, layer=BEST_LAYER, factor=1.2,
                                      label=f"{TASK_ID}__c{BEST_CLUSTER}_mean_f1.2", agg="mean")
    mean_vec = out_mean["vector"]
    all_task_metrics.append({"task": TASK_ID, "group": f"c{BEST_CLUSTER}", "arm": "mean", "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: out_mean["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": out_mean["injected_norm"]})
    # arm 3: top2 of raw rows (norm-matched to mean arm)
    out_top2 = evaluate_with_vectors(cdf, layer=BEST_LAYER, factor=1.2,
                                      label=f"{TASK_ID}__c{BEST_CLUSTER}_top2_f1.2", agg="top2",
                                      sort_key="abs_prob_delta", match_norm_to=mean_vec)
    all_task_metrics.append({"task": TASK_ID, "group": f"c{BEST_CLUSTER}", "arm": "top2", "k": 2,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: out_top2["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": out_top2["injected_norm"]})
    analyze_tokens(base_results, out_top2["results"], f"{TASK_ID}_c{BEST_CLUSTER}_top2")
else:
    print("skipped (no BEST_CLUSTER).")


### Task 7. task_delta_weighted (signed + magnitude weighted means)

In [ ]:
TASK_ID = "task_delta_weighted"
for agg in ("weighted_mean_signed", "weighted_mean_abs"):
    out = evaluate_with_vectors(df, layer=BEST_LAYER, factor=1.2,
                                 label=f"{TASK_ID}__{agg}_f1.2", agg=agg)
    all_task_metrics.append({"task": TASK_ID, "group": "all", "arm": agg, "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": out["injected_norm"]})
    analyze_tokens(base_results, out["results"], f"{TASK_ID}_{agg}")


### Task 8. task_pivot_token_targeted (top-3 pivot tokens by median +prob_delta)

In [ ]:
TASK_ID = "task_pivot_token_targeted"
pos = df[df["prob_delta"] > 0]
top_tokens = (pos.groupby("pivot_token")["prob_delta"].median()
                  .sort_values(ascending=False).head(3).index.tolist())
print("targeted tokens:", top_tokens)
for tok in top_tokens:
    tok_df = df[df["pivot_token"] == tok]
    if len(tok_df) < 2:
        continue
    tok_slug = re.sub(r"\W+", "_", str(tok))[:20]
    mean_out = evaluate_with_vectors(tok_df, layer=BEST_LAYER, factor=1.2,
                                      label=f"{TASK_ID}__{tok_slug}_mean_f1.2", agg="mean")
    mean_vec = mean_out["vector"]
    all_task_metrics.append({"task": TASK_ID, "group": tok_slug, "arm": "mean", "k": None,
                              "layer": BEST_LAYER, "factor": 1.2,
                              **{k: mean_out["metrics"][k] for k in ("accuracy", "f1", "parse_rate")},
                              "injected_norm": mean_out["injected_norm"]})
    analyze_tokens(base_results, mean_out["results"], f"{TASK_ID}_{tok_slug}_mean")
    for k in TOP_K_ARMS:
        arm = f"top{k}"
        out = evaluate_with_vectors(tok_df, layer=BEST_LAYER, factor=1.2,
                                     label=f"{TASK_ID}__{tok_slug}_{arm}_f1.2", agg=arm,
                                     sort_key="abs_prob_delta", match_norm_to=mean_vec)
        all_task_metrics.append({"task": TASK_ID, "group": tok_slug, "arm": arm, "k": k,
                                  "layer": BEST_LAYER, "factor": 1.2,
                                  **{kk: out["metrics"][kk] for kk in ("accuracy", "f1", "parse_rate")},
                                  "injected_norm": out["injected_norm"]})
        analyze_tokens(base_results, out["results"], f"{TASK_ID}_{tok_slug}_{arm}")


## 5e. Comparison + export

In [ ]:
import pandas as pd
metrics_df = pd.DataFrame(all_task_metrics)
metrics_df["base_acc"] = base_metrics["accuracy"]
metrics_df["acc_lift_pp"] = (metrics_df["accuracy"] - base_metrics["accuracy"]) * 100
metrics_df.to_csv(RESULTS_DIR / "all_task_metrics.csv", index=False)
(RESULTS_DIR / "all_task_metrics.json").write_text(metrics_df.to_json(orient="records", indent=2))
metrics_df.sort_values("acc_lift_pp", ascending=False).head(20)


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

# Chart 1: mean vs top1 vs top2 within each (task, group).
multi_arm = metrics_df[metrics_df["arm"].isin(["mean", "top1", "top2"])].copy()
if len(multi_arm):
    multi_arm["task_group"] = multi_arm["task"] + "/" + multi_arm["group"].astype(str)
    piv = multi_arm.pivot_table(index="task_group", columns="arm", values="accuracy")
    fig, ax = plt.subplots(figsize=(12, max(4, 0.3 * len(piv))))
    piv.plot(kind="barh", ax=ax)
    ax.axvline(base_metrics["accuracy"], ls="--", color="gray", label="base")
    ax.set_xlabel("accuracy")
    ax.set_title("mean vs top1 vs top2 by (task, group)")
    ax.legend(); fig.tight_layout()
    fig.savefig(RESULTS_DIR / "arms_per_group.png"); plt.show()

# Chart 2: best arm per task vs base.
best_per_task = metrics_df.sort_values("accuracy", ascending=False).groupby("task").head(1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(best_per_task["task"], best_per_task["accuracy"])
ax.axhline(base_metrics["accuracy"], ls="--", color="gray", label="base")
ax.set_xticklabels(best_per_task["task"], rotation=25, ha="right")
ax.set_ylabel("best accuracy"); ax.set_title("best arm per task vs base")
ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR / "best_per_task.png"); plt.show()

# Chart 3: injected_norm vs accuracy_lift, marker per arm.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for arm, sub in metrics_df.groupby("arm"):
    ax.scatter(sub["injected_norm"], sub["acc_lift_pp"], label=arm, alpha=0.7)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("injected_norm"); ax.set_ylabel("accuracy lift (pp)")
ax.set_title("lift vs injected_norm, by arm")
ax.legend(); fig.tight_layout()
fig.savefig(RESULTS_DIR / "lift_vs_norm.png"); plt.show()


### Interpretation (fill in after running)

- Best aggregation (mean vs top1 vs top2) per task:
- Reasoning patterns with strongest positive lift:
- Cluster_vector vs mean-of-raw:
- Signed-delta prediction (pos helps / neg hurts): confirmed? ...

## 5f. Promoted-words aggregate

In [ ]:
agg_rows = []
for state_file in (TASKS_ROOT).rglob("token_analysis.json"):
    d = json.loads(state_file.read_text())
    for w, lr, fb, fs in d.get("promoted", [])[:3]:
        agg_rows.append({"task_id": d["task_id"], "word": w, "log_ratio": lr})
promoted_df = pd.DataFrame(agg_rows)
promoted_df.to_csv(RESULTS_DIR / "promoted_words_by_task.csv", index=False)
print(f"Aggregated {len(promoted_df)} promoted-word rows -> promoted_words_by_task.csv")


## 6. Bundle everything

In [ ]:
zip_path = Path(f"nb_results_{RUN_TAG}.zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(RESULTS_DIR.parent))
print(f"Zipped {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Not on Colab — the zip is on the local filesystem.")
